In [1]:
!pwd

/root/jupyter_notebooks/FYP


In [2]:
import numpy as np
from scipy import signal
import tensorflow as tf
from tensorflow.keras import layers, Model, initializers, Sequential
from optic.models.devices import mzm, photodiode, edfa, iqm, coherentReceiver, pdmCoherentReceiver, basicLaserModel
from optic.models.channels import linearFiberChannel, ssfm
from optic.comm.modulation import modulateGray, grayMapping
from optic.comm.sources import bitSource, symbolSource
from optic.dsp.core import upsample, pulseShape, pnorm, anorm, signalPower, firFilter, decimate, symbolSync,phaseNoise

try:
    from optic.dsp.coreGPU import checkGPU
    if checkGPU():
        from optic.dsp.coreGPU import firFilter
    else:
        from optic.dsp.core import firFilter
except ImportError:
    from optic.dsp.core import firFilter

from optic.utils import parameters, dBm2W, ber2Qfactor
from optic.plot import eyediagram, pconst, plotPSD
import matplotlib.pyplot as plt
from scipy.special import erfc
from tqdm.notebook import tqdm
import scipy as sp
import scipy.constants as const

try:
    from optic.models.modelsGPU import manakovSSF
except:
    from optic.models.channels import manakovSSF

from optic.dsp.equalization import edc, mimoAdaptEqualizer, ffe
from optic.dsp.carrierRecovery import cpr
from optic.comm.metrics import fastBERcalc, monteCarloGMI, monteCarloMI, calcEVM, bert
from optic.dsp.clockRecovery import gardnerClockRecovery


import logging as logg
logg.basicConfig(level=logg.INFO, format='%(message)s', force=True)
import time
from helper_funcs import *

In [3]:
from pynq import Overlay
from pynq import allocate
import numpy as np
overlay = Overlay('vivado_export.xsa')

In [4]:
from pynq import ps

print(ps.Clocks.fclk0_mhz)
# ps.Clocks.fclk0_mhz = 375
print(ps.Clocks.fclk0_mhz)
print(ps.Clocks.cpu_mhz)

249.9975
249.9975
1333.32


In [5]:
ip = overlay.MultilayerPerceptron_0
mmio = ip.mmio
register_map = ip.register_map
registers = register_map._register_classes

In [6]:
for name, reg in registers.items():
    print(name, reg)

CTRL (<class 'pynq.registers.RegisterCTRL'>, 0, 32, None, None, 'read-write')
GIER (<class 'pynq.registers.RegisterGIER'>, 4, 32, None, None, 'read-write')
IP_IER (<class 'pynq.registers.RegisterIP_IER'>, 8, 32, None, None, 'read-write')
IP_ISR (<class 'pynq.registers.RegisterIP_ISR'>, 12, 32, None, None, 'read-write')
sigI_in_1 (<class 'pynq.registers.RegistersigI_in_1'>, 16, 32, None, None, 'write-only')
sigI_in_2 (<class 'pynq.registers.RegistersigI_in_2'>, 20, 32, None, None, 'write-only')
sigQ_in_1 (<class 'pynq.registers.RegistersigQ_in_1'>, 28, 32, None, None, 'write-only')
sigQ_in_2 (<class 'pynq.registers.RegistersigQ_in_2'>, 32, 32, None, None, 'write-only')
sigI_out_1 (<class 'pynq.registers.RegistersigI_out_1'>, 40, 32, None, None, 'write-only')
sigI_out_2 (<class 'pynq.registers.RegistersigI_out_2'>, 44, 32, None, None, 'write-only')
sigQ_out_1 (<class 'pynq.registers.RegistersigQ_out_1'>, 52, 32, None, None, 'write-only')
sigQ_out_2 (<class 'pynq.registers.RegistersigQ_ou

In [7]:
# Allocated buffer (m_axi)
no_symbols = 5000
input_buffer_size = no_symbols
output_buffer_size = no_symbols

sigI_in_buffer = allocate(shape=(no_symbols,), dtype=np.float32)
sigQ_in_buffer = allocate(shape=(no_symbols,), dtype=np.float32)
sigI_out_buffer = allocate(shape=(no_symbols,), dtype=np.float32)
sigQ_out_buffer = allocate(shape=(no_symbols,), dtype=np.float32)


In [8]:
register_map.sigI_in_1.sigI_in = sigI_in_buffer.device_address # no need for the upper 32bits
register_map.sigQ_in_1.sigQ_in = sigQ_in_buffer.device_address # no need for the upper 32bits
register_map.sigI_out_1.sigI_out = sigI_out_buffer.device_address # no need for the upper 32bits
register_map.sigQ_out_1.sigQ_out = sigQ_out_buffer.device_address # no need for the upper 32bits


In [9]:
def build_model():
    inputs = layers.Input(shape=(None, 2)) # 2 for I and Q

    sec_a = layers.Conv1D(2, 101, padding='same')(inputs) # 100 taps was a sweet spot, 20-ish fails to converge, tiker with different values.

    nonlinear_1 = layers.Dense(20, activation=tf.math.sin)(sec_a)
    nonlinear_2 = layers.Dense(20, activation=tf.math.sin)(nonlinear_1)
    nonlinear_3 = layers.Dense(2, activation='linear')(nonlinear_2)
    
    outputs = layers.Add()([sec_a, nonlinear_3]) 
    
    model = Model(inputs, outputs)
    model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=1e-2), loss='mse') # ILA)
    model.load_weights("best_model.weights.h5")
    return model


In [10]:
# Hardware accelerated function
def dpd_hw(symbTx_nn):
    symbTx_r = symbTx_nn[:, 0]
    symbTx_im = symbTx_nn[:, 1]
    # Write to input buffer
    sigI_in_buffer[:len(symbTx_r)] = symbTx_r
    sigQ_in_buffer[:len(symbTx_r)] = symbTx_im

    # Send start signal
    register_map.CTRL.AP_START = 1
    
    # Wait until algorithm has completed
    while (register_map.CTRL.AP_DONE == 0):
        pass

In [11]:
M = 16
no_symbols= 100_000 # must be multiple of 5000 (seq_length)
nBits = int(no_symbols * np.log2(M))
SpSout = 2
mzmScale = 0.8
laserLinewidth = 100e3
dpd_model = build_model()

paramSymb = intialise_paramSymb(M, nBits, seed=333)
symbTx = symbolSource(paramSymb)

symbTx_nn = preprocess(symbTx) # shape = 20,5000,2

# dpd_model.predict()
single_batch = symbTx_nn[0][np.newaxis, ...] # shape = 1,5000,2

sw_time = %timeit -r 10 -o dpd_model.predict(single_batch, verbose=0)
hw_time = %timeit -r 10 -o dpd_hw(single_batch[0])

print(f"AVG Execution Time - CPU: {sw_time.average}")
print(f"AVG Execution Time - w/ FPGA Acceleration: {hw_time.average}")
print('Performance gain:', sw_time.average / hw_time.average)


/usr/local/share/pynq-venv/lib/python3.10/site-packages/keras/src/saving/saving_lib.py:797: UserWarning: Skipping variable loading for optimizer 'adam', because it has 2 variables whereas the saved optimizer has 18 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))


313 ms ± 1.55 ms per loop (mean ± std. dev. of 10 runs, 1 loop each)
251 µs ± 566 ns per loop (mean ± std. dev. of 10 runs, 1,000 loops each)
AVG Execution Time - CPU: 0.3131848199996966
AVG Execution Time - w/ FPGA Acceleration: 0.0002512434404998203
Performance gain: 1246.5392902463484


In [ ]:
## Testing BER With HW Output:
symbDPD_hw =[]
for batch in symbTx_nn:
    dpd_hw(batch)
    symbDPD_hw.append(sigI_out_buffer + 1j*sigQ_out_buffer)

symbDPD_hw = np.array(symbDPD_hw).flatten()
symbDPD_sw = merge_i_q(dpd_model.predict(symbTx_nn)).flatten()


## SW BER
print("PERF CALC W/ TF DPD")
y_CPR_1, d, phaseEst = simulate_optical_system(symbDPD_sw, len(symbDPD_sw), M, PA_enable=True, Data_Aided=True, SpSout=SpSout, mzmScale=mzmScale)
perf_calc(symbTx, y_CPR_1, d, M, paramSymb)


## HW BER
print("PERF CALC W/ FPGA-accelerated DPD")
y_CPR_1, d, phaseEst = simulate_optical_system(symbDPD_hw, len(symbDPD_hw), M, PA_enable=True, Data_Aided=True, SpSout=SpSout, mzmScale=mzmScale)
perf_calc(symbTx, y_CPR_1, d, M, paramSymb)